In [ ]:
%uv pip install -q langchain-groq langchain-core requests

In [3]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [79]:
# tool creating

@tool
def multiply(a:int,b:int) ->int:
    """Give 2 number a and b this tool return theire product"""
    return a *b

print(multiply.name)

multiply


In [80]:
multiply.invoke({'a':3,'b':4})

12

In [81]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [82]:
multiply.description

'Give 2 number a and b this tool return theire product'

#### tool calling

In [ ]:
# tool binding with llm

llm = ChatGroq(
    model= "llama-3.3-70b-versatile",
    temperature=0.7
)

In [84]:
querry = HumanMessage('can you multiply 3 with 1000')

In [85]:
messages = [querry]
messages


[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]

In [86]:
llm_with_tool = llm.bind_tools([multiply])

In [87]:
llm_with_tool.invoke('hi how are you')

AIMessage(content="I'm just a language model, I don't have feelings or emotions, but I'm here to help you with any questions or tasks you may have. Is there something specific you'd like to talk about or ask for help with?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 236, 'total_tokens': 284, 'completion_time': 0.117366825, 'completion_tokens_details': None, 'prompt_time': 0.041034421, 'prompt_tokens_details': None, 'queue_time': 0.161203567, 'total_time': 0.158401246}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cc797-ac64-76e0-8a15-6e8e963dae5f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 236, 'output_tokens': 48, 'total_tokens': 284})

In [88]:
result = llm_with_tool.invoke(messages)

In [89]:
messages.append(result)
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'azhq2am8z', 'function': {'arguments': '{"a":3,"b":1000}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 241, 'total_tokens': 261, 'completion_time': 0.057457452, 'completion_tokens_details': None, 'prompt_time': 0.012281913, 'prompt_tokens_details': None, 'queue_time': 0.049204377, 'total_time': 0.069739365}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cc797-ae4f-7ba1-81f0-8af707ff21a0-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'azhq2am8z', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 241, 'output_tokens': 20, 'total_tokens': 261})]

In [90]:
result.tool_calls[0]['args']

{'a': 3, 'b': 1000}

##### tool execution

In [91]:
multiply.invoke(result.tool_calls[0])

ToolMessage(content='3000', name='multiply', tool_call_id='azhq2am8z')

In [92]:
result.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 3, 'b': 1000},
 'id': 'azhq2am8z',
 'type': 'tool_call'}

In [93]:
multiply.invoke({'name': 'multiply',
 'args': {'a': 3, 'b': 10},
 'id': 'brqafa5h6',
 'type': 'tool_call'})

ToolMessage(content='30', name='multiply', tool_call_id='brqafa5h6')

In [94]:
tool_result = multiply.invoke(result.tool_calls[0])

In [95]:
messages.append(tool_result)
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'azhq2am8z', 'function': {'arguments': '{"a":3,"b":1000}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 241, 'total_tokens': 261, 'completion_time': 0.057457452, 'completion_tokens_details': None, 'prompt_time': 0.012281913, 'prompt_tokens_details': None, 'queue_time': 0.049204377, 'total_time': 0.069739365}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cc797-ae4f-7ba1-81f0-8af707ff21a0-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'azhq2am8z', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 241, 'output_tokens': 20, 'total_tokens': 261}),
 Too

In [96]:
llm_with_tool.invoke(messages).content

'The result of multiplying 3 by 1000 is 3000.'

In [1]:
# 9d13047e18-3ef2c2a9f2-tbiw0f

In [8]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'
  
  response = requests.get(url)
  
  return response.json()



@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [9]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [7]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1772841601,
 'time_last_update_utc': 'Sat, 07 Mar 2026 00:00:01 +0000',
 'time_next_update_unix': 1772928001,
 'time_next_update_utc': 'Sun, 08 Mar 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 91.9215}

In [10]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999